In [ ]:

# %pip install bertopic umap-learn

In [1]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from gensim.corpora import Dictionary, MmCorpus
from sklearn.preprocessing import normalize
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.parsing.preprocessing import STOPWORDS
import math
import umap.umap_ as umap 
from hdbscan import HDBSCAN


d:\TB2\Intro to AI and TEXT Analytics (EMATM0067)\Coursework\TextAnalytics-CW-Task4\venv3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csv_file = Path("../data/customer_support_tickets_preprocessed.csv")
df = pd.read_csv(csv_file)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket Description charlength,Ticket Description wordlength,Tfidf_ticket_description,Embeddings_ticket_description
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN,290,43,billing code appreciate requested website addr...,i'm having an issue with the . please assist. ...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN,288,44,existing intermittent unexpectedly,i'm having an issue with the . please assist. ...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,277,42,turning yesterday respond original charger cha...,i'm facing a problem with my . the is not turn...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,264,41,interested love happen check feedback multiple,i'm having an issue with the . please assist. ...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,336,55,note seller responsible damage arising deliver...,i'm having an issue with the . please assist. ...


In [3]:
df.shape

(8044, 21)

In [4]:
# Load the TF-IDF feature matrix 

X_tfidf = joblib.load("../data/tfidf_sklearn.pkl")
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 119519 stored elements and shape (8044, 1855)>

In [5]:
# Load the sentence embeddings

X_embed = np.load("../data/sentence_embeddings.npy")
X_embed

array([[-0.02683849, -0.02634216,  0.03688251, ...,  0.03251351,
        -0.04797408, -0.04766009],
       [-0.06192231, -0.06736261,  0.07069816, ...,  0.03437989,
         0.03009921,  0.03516207],
       [-0.06530187, -0.01148537,  0.06769149, ..., -0.03189831,
         0.0032761 ,  0.01859518],
       ...,
       [ 0.01327254, -0.10080557,  0.04425666, ...,  0.07094863,
        -0.02432872,  0.01956789],
       [-0.05345704, -0.07887096,  0.00049884, ...,  0.00188731,
         0.01710037,  0.08331525],
       [-0.00896007, -0.03504539,  0.08467014, ..., -0.04020412,
         0.04176571, -0.0434691 ]], shape=(8044, 384), dtype=float32)

In [6]:
tfidf_vectorizer = TfidfVectorizer(max_df = 0.70, min_df = 5, ngram_range = (1,3))
tfidf_vectorizer.fit_transform(df['Tfidf_ticket_description'])

# Extract feature names learned by TF-IDF vectorizer
feature_names = tfidf_vectorizer.get_feature_names_out()
feature_names

array(['ability', 'absolutely', 'accept', ...,
       'yesterday respond software', 'yesterday respond solution', 'york'],
      shape=(1855,), dtype=object)

In [7]:
print("Size of vocabulary : " , len(feature_names))

Size of vocabulary :  1855


In [8]:
# Stopwords

custom_stopwords = {
    'please', 'help', 'assist', 'support', 'thanks', 'thank','soon','mentioned',
    'im', 'ive', 'us','would', 'could', 'need', 'want', 'trying',
    'tried','check', 'checked', 'make', 'made', 'get', 'getting','also',
    'use', 'using', 'used','thing', 'something', 'anything', 'everything',
    'way', 'time','issue', 'problem', 'request', 'work', 'working', 'fine',
    'available', 'recent', 'recently','facing', 'doe', 'noticed', 'happening',
    'started', 'happen','different', 'steps', 'did', 'regards','already', 'multiple',
    'last','times','followed', 'reviewed','specific', 'possible', 'related','new',
    'old','find', 'try', 'say', 'mean','name', 'email', 'price', 'one', 'unresolved',
    'add','note', 'may', 'dont', 'know','sure', 'changes', 'performed', 'properly',
    'original','like', 'similar','reported','doesnt', 'sometimes', 'acts', 'works',
    'ensure', 'desired', 'action', 'remains', 'life', 'seems', 'might', 'guide',
    'much', 'others', 'heavily', 'daily', 'task', 'affecting', 'assistance','hoping',
    'persists','didnt','option', 'perform', 'recommendation', 'information', 'official',
    'solution', 'provide', 'making', 'user', 'customer', 'item', 'device','far', 'luck',
    'contact', 'contacted', 'occurring','resolve', 'function', 'came', 'having', 'change',
    'haven', 'let', 'unable', 'able', 'afterward', 'var', 'step', 'order', 've', 'll', 'disqus', 'comments', 'javascript'
}
final_stopwords = STOPWORDS.union(custom_stopwords)


## KMeans

In [ ]:
# Define a function to plot the elbow curve to identify optimal K value

def plot_elbow(text_representation, low_k = 2, high_k = 20, plot_title = "Elbow Method for optimal k"):
    inertia = []
    k_range = range(low_k, high_k)

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, init='k-means++')
        
        km.fit(text_representation)
        inertia.append(km.inertia_)

    plt.plot(k_range, inertia, marker='o')
    plt.title(plot_title)
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Inertia")
    plt.show()

In [ ]:
# Define a function to plot the silhouette scores for various K value

def plot_silhouette_scores(text_representation, low_k = 2, high_k = 20, plot_title = "Silhouette score"):
    silhouette_scores = []
    k_range = range(low_k, high_k)

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km_labels = km.fit_predict(text_representation)

        score = silhouette_score(text_representation, km_labels)
        silhouette_scores.append(score)

    plt.plot(k_range, silhouette_scores, marker='o')
    plt.title(plot_title)
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Silhouette Scores")
    plt.show()

In [ ]:
def plot_cluster_distribution(df_results, plot_title = "Cluster Distribution Plot"):
    num_plots = len(df_results)
    cols = 3
    rows = math.ceil(num_plots/cols)

    fig, axes = plt.subplots(rows, cols, figsize = (20, 5* rows))

    axes = axes.flatten()

    for i, (j, row) in enumerate(df_results.iterrows()):

        ax = axes[i]

        k = row['Number of Clusters (k) ']
        silhouette_score = row['Silhouette Score ']
        cluster_distribution = row['Distribution of Clusters ']

        cluster_id = list(cluster_distribution.keys())
        distribution = list(cluster_distribution.values())

        bars = ax.bar(cluster_id, distribution)

        for id in range(len(cluster_id)):
            ax.text (cluster_id[id], distribution[id] + 5, f"{distribution[id]}", ha = "center", fontsize = 9)

        # plt.text(cluster_id[i], distribution[i]+2, f"{distribution[i]}", ha = "center")

        ax.set_title(f"k = {k}, score = {silhouette_score:.2f}", fontsize=12, fontweight='bold')
        ax.set_xticks(cluster_id)

    fig.suptitle(plot_title, fontsize = 18)
    plt.show()

In [ ]:
# Tabulate the number of clusters and its silhouette score along with its distribution between clusters

def kmeans_cluster_evaluation(text_representation, low_k = 2, high_k = 20):
    res = []
    k_range = range(low_k, high_k)

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km_labels = km.fit_predict(text_representation)

        score = silhouette_score(text_representation, km_labels)
        
        cluster_distribution = pd.Series(km_labels).value_counts().sort_index().to_dict()

        centroids = km.cluster_centers_

        cluster_words = {}
        for i in range(k):
            indices = centroids[i].argsort()[::-1][:30]
            words = [feature_names[j] for j in indices]
            cluster_words[f"Cluster {i}"] = ', '.join(words)

        res.append({
            'Number of Clusters (k) ' : k,
            'Silhouette Score ' : round(score, 2),
            'Distribution of Clusters ' : cluster_distribution,
            'Top n Words per cluster' : cluster_words
        })

    df_res = pd.DataFrame(res)

    return df_res

### KMeans + TFIDF

In [ ]:
df_res = kmeans_cluster_evaluation(X_tfidf, 2, 20)

In [ ]:
# Plot the distribution of words between clusters

plot_cluster_distribution(df_res, "Cluster Distribution Plot - KMeans + TFIDF")

In [ ]:
# Plot the elbow curve for k range (2,10)

plot_elbow(X_tfidf, 2, 20, "Elbow Method for optimal k - TFIDF")

In [ ]:
# Plot the silhouette scores for k range (2,10)

plot_silhouette_scores(X_tfidf, 2, 20, "Silhouette score (KMeans + TFIDF)")

In [ ]:
filepath = Path("../Tests/kmeans_tfidf_1.txt")

with open(filepath, "w") as file:
    for k,v in df_res["Top n Words per cluster"].items():
        file.write(f"{k + 2} clusters \n")
        file.write(str(v))
        file.write("\n\n"+"-" * 40 +"\n\n")

#### SVD

In [ ]:
n_comp = 50

# Apply Truncated SVD to reduce high dimensional TF-IDF feature space
X_tfidf_svd = TruncatedSVD(n_components= n_comp)

# Fit the SVD model - transform sparse TF-IDF features into dense, low dimensional representation
X_tfidf_reduced = X_tfidf_svd.fit_transform(X_tfidf)

# Normalise the reduced vectors
X_tfidf_normalized = Normalizer().fit_transform(X_tfidf_reduced)

# calculate cumulative explained variance to understance how much information is retained
cumulative_variance = X_tfidf_svd.explained_variance_ratio_.cumsum()

# Plot cumulative variance to select optimal number of dimensions
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='.')
plt.axvline(x = 50, color = 'r', linestyle = '--', label = f"Variance = {cumulative_variance[n_comp - 1] * 100 : .2f} % \n(n_components = 200) ")
plt.title("Amount of information retained vs Number of dimensions")
plt.xlabel("Number of SVD components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()

In [ ]:
df_res = kmeans_cluster_evaluation(X_tfidf_normalized, 2, 20)

In [ ]:
# Plot the distribution of words between clusters

plot_cluster_distribution(df_res, "Cluster Distribution Plot - KMeans + TFIDF + SVD")

In [ ]:
# Plot the elbow curve for k range (2,20)

plot_elbow(X_tfidf_normalized, 2, 20,"Elbow Method for optimal k - TFIDF (SVD)")

In [ ]:
# Plot the elbow curve for k range (2,50)

# plot_elbow(X_tfidf_normalized, 2, 50)

In [ ]:
# Plot the silhouette scores for k range (2,10)

plot_silhouette_scores(X_tfidf_normalized, 2, 20, "Silhouette score (KMeans + TFIDF + SVD)")

Selecting optimal k-value as 6, tested kmeans algorithm using anywhere between 2 to 10 and evaluated its performance using Silhoutte score

The elbow plot shows a noticeable redn in inertia up to k = 6, after which rate of improvement slows which indicates that increase in no of clusters diminishes the returns.

The silhoutte score continues to increase with higher k values, the overall scores remain relatively low, suggesting that clusters are not strongly seperated which is expected when using tfidf repn as it captures lexical similarity rather than deeper semantic relationships which is the motivation to experiment with embeddings approach further.

selecting a very high k vakue may lead to overly specific clusters which are difficult to interpret as meaningful customer issue categories.

Hence, k = 6 was selected as a balance between cluster compactness and interpretability, avoiding over segmentation of issue categories.

In [ ]:
filepath = Path("../Tests/kmeans_tfidf_svd_1.txt")


with open(filepath, "w") as file:
    for k,v in df_res["Top n Words per cluster"].items():
        file.write(f"{k + 2} clusters \n")
        file.write(str(v))
        file.write("\n\n"+"-" * 40 +"\n\n")

In [ ]:
pd.set_option('display.max_colwidth', None)

#### UMap

In [ ]:
umap_reducer = umap.UMAP(n_neighbors=50, n_components= 100, metric='cosine', random_state=42)
X_tfidf_umap = umap_reducer.fit_transform(X_tfidf)

In [ ]:
df_res = kmeans_cluster_evaluation(X_tfidf_umap, 2, 20)

In [ ]:
# Plot the distribution of words between clusters

plot_cluster_distribution(df_res, "Cluster Distribution Plot - KMeans + TFIDF + UMap")

In [ ]:
# Plot the elbow curve for k range (2,20)

plot_elbow(X_tfidf_umap, 2, 20,"Elbow Method for optimal k - TFIDF (UMap)")

In [ ]:
# Plot the silhouette scores for k range (2,10)

plot_silhouette_scores(X_tfidf_umap, 2, 20, "Silhouette score (KMeans + TFIDF + UMap)")

In [ ]:
filepath = Path("../Tests/kmeans_tfidf_umap_1.txt")


with open(filepath, "w") as file:
    for k,v in df_res["Top n Words per cluster"].items():
        file.write(f"{k + 2} clusters \n")
        file.write(str(v))
        file.write("\n\n"+"-" * 40 +"\n\n")

The silohouette score for k = 8 and more is tend to be 0.086 however the distribution between clusterd tends to be that it dumped almost half of all the tickets into one massive pile (cluster 0 - 3127 tickets) which is not helpful for a customer support team to fix specific problems.

At k=6, score is less comparatively which is 0.064 but there is an balanced distribution.

### KMeans + Embeddings

In [ ]:
X_embed_normalized = normalize(X_embed)

In [ ]:
df_res = kmeans_cluster_evaluation(X_embed_normalized, 2, 20)

In [ ]:
# Plot the distribution of words between clusters

plot_cluster_distribution(df_res, "Cluster Distribution Plot - KMeans + Embeddings")

In [ ]:
plot_elbow(X_embed_normalized, 2, 20, "Elbow Method for optimal k - Embeddings")

In [ ]:
# plot_elbow(X_embed_normalized, 2, 50)

In [ ]:
plot_silhouette_scores(X_embed_normalized, 2, 20, "Silhouette score (KMeans + Embeddings)")

## LDA

In [ ]:
# Load the dictionary and corpus files

dictionary = Dictionary.load("../data/gensim_lda_dictionary.dict")
bow_corpus = MmCorpus("../data/gensim_lda_bow_corpus.mm")
tfidf_corpus = MmCorpus("../data/gensim_lda_tfidf_corpus.mm")

In [ ]:
docs = df['Tfidf_ticket_description'].tolist()
tokenised_docs = [text.split() for text in df['Tfidf_ticket_description']]

### TFIDF - LDA

In [ ]:
# Train the LDA model using the TF-IDF corpus

lda_model_tfidf = LdaModel(
    corpus= tfidf_corpus,
    num_topics= optimal_k,
    id2word= dictionary,
    passes= 10,
    random_state=42
)

In [ ]:
# Print the topics along with words and their probabilities discovered by the LDA model using the TF-IDF corpus

for idx, topic in lda_model_tfidf.print_topics(-1):
    print(f"Topic {idx} : {topic}\n")

In [ ]:
# Calculate the coherence score for the LDA model using the TF-IDF corpus

coherence_umass_lda = CoherenceModel(
    model = lda_model_tfidf,
    corpus = tfidf_corpus,
    dictionary = dictionary,
    coherence= 'u_mass'
)

coherence_score_tfidf_umass = coherence_umass_lda.get_coherence()
coherence_score_tfidf_umass

In [ ]:
# Calculate the coherence score for the LDA model using the TF-IDF corpus

coherence_cv_lda = CoherenceModel(
    model = lda_model_tfidf,
    corpus = tfidf_corpus,
    texts = tokenised_docs,
    coherence= 'c_v'
)

coherence_score_tfidf_cv = coherence_cv_lda.get_coherence()
coherence_score_tfidf_cv

### BOW - LDA

In [ ]:
# Train the LDA model using the Bag of words corpus

lda_model_bow = LdaModel(
    corpus= bow_corpus,
    num_topics= optimal_k,
    id2word= dictionary,
    passes= 10,
    random_state=42
)

In [ ]:
for idx, topic in lda_model_bow.print_topics(-1):
    print(f"Topic {idx} : {topic}\n")

In [ ]:
# Calculate the coherence score using u_mass coherence for the LDA model using the BOW corpus

coherence_umass_bow = CoherenceModel(
    model = lda_model_bow,
    corpus = bow_corpus,
    dictionary = dictionary,
    coherence= 'u_mass'
)

coherence_score_bow = coherence_umass_bow.get_coherence()
coherence_score_bow

In [ ]:
# Calculate the coherence score using c_v coherence for the LDA model using the BOW corpus

coherence_cv_bow = CoherenceModel(
    model = lda_model_bow,
    corpus = bow_corpus,
    texts= tokenised_docs,
    coherence= 'c_v'
)

coherence_score_bow = coherence_cv_bow.get_coherence()
coherence_score_bow

## HAC

### HAC - TFIDF

In [ ]:

z = linkage(X_tfidf.toarray(), method='ward')

dendrogram(z, truncate_mode='lastp', p = 20)
plt.xticks(rotation=90)

In [ ]:
scores = []

k_range = range(2,10)

for k in k_range:
    hac = AgglomerativeClustering(n_clusters=k, linkage = 'ward')
    hac_labels = hac.fit_predict(X_tfidf.toarray())

    score = silhouette_score(X_tfidf, hac_labels)
    scores.append(score)

plt.plot(k_range, scores, marker='o')
plt.title("Silhouette score for HAC")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")


In [ ]:
optimal_k_hac = 6

hac = AgglomerativeClustering(n_clusters=optimal_k_hac, linkage = 'ward')
hac_labels = hac.fit_predict(X_tfidf.toarray())

df['hac_tfidf_labels'] = hac_labels

df['hac_tfidf_labels']

In [ ]:
hac_tfidf_dist = df['hac_tfidf_labels'].value_counts().sort_index()
hac_tfidf_dist.plot(kind='bar')


plt.title("Cluster Distribution (HAC - TFIDF)")
plt.xlabel("Cluster")
plt.ylabel("Number of tickets")

for i,j in enumerate(hac_tfidf_dist):
    plt.text(i, j+2, f"{hac_tfidf_dist.iloc[i]}", ha = "center")



In [ ]:
hac_tfidf_dist

In [ ]:
silhouette_score(X_tfidf, df['hac_tfidf_labels'])

In [ ]:
for cluster_num in range(optimal_k):
    cluster_index = np.where(df['hac_tfidf_labels'] == cluster_num)[0]
    mean_tfidf_hac = np.asarray(X_tfidf[cluster_index].mean(axis = 0)).flatten()

    ordered_words = [feature_names[i] for i in mean_tfidf_hac.argsort()[::-1][:30]]

    print(f"Cluster {cluster_num} : {', '.join(ordered_words)}")

### HAC - Embedding

In [ ]:
z = linkage(X_embed, method='ward')

dendrogram(z, truncate_mode='lastp', p = 20)
plt.xticks(rotation=90)

In [ ]:
scores = []

k_range = range(2,10)

for k in k_range:
    hac = AgglomerativeClustering(n_clusters=k, linkage = 'ward')
    hac_labels = hac.fit_predict(X_embed)

    score = silhouette_score(X_embed, hac_labels)
    scores.append(score)

plt.plot(k_range, scores, marker='o')
plt.title("Silhouette score for HAC")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")


In [ ]:
optimal_k_hac = 6

hac = AgglomerativeClustering(n_clusters=optimal_k_hac, linkage = 'ward')
hac_embed_labels = hac.fit_predict(X_embed)

df['hac_embed_labels'] = hac_embed_labels

df['hac_embed_labels']

## BertTopic

In [9]:
def metrics_calculation(vectorized_model, bert_model):

    df['Bertopic_labels'] = bert_model.topics_
    # Calculate metric 1 - outlier percentage

    total_customer_tikets = len(df)
    outlier_customer_tickets = len(df[df['Bertopic_labels'] == -1])

    outlier_percentage = outlier_customer_tickets / total_customer_tikets * 100
   
    analyser = vectorized_model.build_analyzer()

    docs = df['Embeddings_ticket_description'].tolist()
    tokenised_docs = [analyser(doc) for doc in docs]

    dictionary = Dictionary(tokenised_docs)
    bow_corpus = [dictionary.doc2bow(doc) for doc in tokenised_docs]

    # Get the topic and its words

    bertopic_words = []
    topic_words = {}
    bertopic_dict = bert_model.get_topics()

    for bertopic_id, scores in bertopic_dict.items():
        # skip the outlier topic
        if bertopic_id == -1:  
            continue

        words = [word for word, score in scores]
        bertopic_words.append(words)

        topic_words[f"Topic {bertopic_id}"] = ', '.join(words)

    # Metric 2 - Calculate topic diversity

    unique_bertopic_words = set()
    total_words = 0
    for word in bertopic_words:
        unique_bertopic_words.update(word)
        total_words += len(word)

    diversity_score = len(unique_bertopic_words) / total_words

    # Metric 3 - Calculate topic coherence using c_v measure

    coherence_model = CoherenceModel(
        topics = bertopic_words,
        texts = tokenised_docs,
        corpus= bow_corpus,
        dictionary= dictionary,
        coherence= 'c_v'
    )

    coherence_score = coherence_model.get_coherence()
        
    return diversity_score, coherence_score, outlier_percentage, topic_words 

Tested through experimentation of parameter combinations ranging from min_cluster_size = 20 and min_samples = 5 to min_cluster_size = 100 and min_samples = 15, from which min_cluster_size = 50 and min_samples = 7 is identified to be optimal threshold for this dataset.

In [22]:
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples = 7, metric='euclidean', cluster_selection_method='leaf', prediction_data= True)

The (1,3) N-gram range is selected as it prevents natural language structure. Models generating 11,15,or 18 topics achieved slightly higher coherance scores, but after qualitative inspection it revealed that 13 topics gave the exact semantic cut required to map clusters to real world business departments. 11 topics caused distinct issues to merge, while 18 topics fragmented core departments into unmanageable sub divisions.

| Diversity Score | Coherence Score ($c_v$) | Outlier Percentage | Topics | N-gram Range |
| :--- | :--- | :--- | :--- | :--- |
| 0.94 | 0.73 | 15.80% | 12 | (1,3) |
| 0.96 | 0.85 | 21.67% | 18 | (1,3) |
| 0.96 | 0.84 | 21.67% | 15 | (1,3) |
| **0.95** | **0.82** | **20.67%** | **13** | **(1,3)** |
| 0.95 | 0.86 | 21.67% | 11 | (1,3) |
| 0.94 | 0.84 | 21.67% | 10 | (1,3) |
| 1.00 | 0.90 | 19.17% | 12 | (2,3) |
| 1.00 | 0.88 | 20.13% | 18 | (2,3) |
| 1.00 | 0.91 | 20.13% | 14/13 | (2,3) |
| 1.00 | 0.85 | 20.13% | 10 | (2,3) |
| 0.93 | 0.81 | 18.19% | 18 | (1,2) |
| 0.94 | 0.81 | 18.19% | 16 | (1,2) |
| 0.93 | 0.79 | 18.19% | 14 | (1,2) |
| 0.94 | 0.78 | 18.19% | 13 | (1,2) |
| 0.93 | 0.74 | 18.19% | 11 | (1,2) |
| 0.98 | 0.73 | 20.43% | 18 | (3,3) |
| 0.98 | 0.68 | 20.43% | 16 | (3,3) |
| 1.00 | 0.64 | 20.43% | 13 | (3,3) |

Therefore, (1,3) N-gram range and 13-topic model provided the optimal balance of mathematical stability andoperational routing.

In [23]:
vectorized_model = CountVectorizer(stop_words = list(final_stopwords), ngram_range = (1,3))

bert_model = BERTopic(embedding_model="all-MiniLM-L6-v2", vectorizer_model= vectorized_model, hdbscan_model=hdbscan_model, verbose=True)
topics, probs = bert_model.fit_transform(df['Embeddings_ticket_description'].tolist())

2026-04-18 21:13:18,228 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5153.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 252/252 [06:11<00:00,  1.48s/it]
2026-04-18 21:20:26,378 - BERTopic - Embedding - Completed ✓
2026-04-18 21:20:26,381 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-18 21:20:41,585 - BERTopic - Dimensionality - Completed ✓
2026-04-18 21:20:41,589 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-18 21:20:42,652 - BERTopic - Cluster - Completed ✓
2026-04-18 21:20:42,684 - BERTopic - Representation - Fine-tuning topics using representation 

In [24]:
# bert_model.visualize_hierarchy()

In [25]:
# In BERTopic, -1 is considered as outlier or Noise category.

num_topics = len(bert_model.get_topic_info()) - 1
num_topics

51

In [26]:
# Reduce the number of topics to 7 

bert_model.reduce_topics(df['Embeddings_ticket_description'], nr_topics=14)

2026-04-18 21:20:46,732 - BERTopic - Topic reduction - Reducing number of topics
2026-04-18 21:20:46,788 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-04-18 21:20:48,819 - BERTopic - Representation - Completed ✓
2026-04-18 21:20:48,830 - BERTopic - Topic reduction - Reduced number of topics from 52 to 14


In [27]:
# bert_model.get_topic_info()

In [28]:
diversity_score, coherence_score, outlier_percentage, topic_words  = metrics_calculation(vectorized_model, bert_model)

In [29]:
print(f"\n Diversity Score : {diversity_score:.2f} \n Coherence Score : {coherence_score:.2f} \n Outlier Percentage : {outlier_percentage:.2f}")


 Diversity Score : 0.95 
 Coherence Score : 0.82 
 Outlier Percentage : 20.67


Topic 0 - Firmware and general updates \
Topic 1 - Software bugs and App crashes \
Topic 2 - Data Recovery \
Topic 3 - Identify and Account access \
Topic 4 - Network setup \
Topic 5 - UI errors \
Topic 6 - Privacy and data security \
Topic 7 - Purchase management \
Topic 8 - Display and screen issues \
Topic 9 - Internet stability \
Topic 10 - Mechanical and internal failures \
Topic 11 - Unresponsive devices \
Topic 12 - Power and charging issues

In [30]:
topic_words

{'Topic 0': 'product, update, software, troubleshooting, software update, productivity, updated, updated firmware, firmware, updated firmware update',
 'Topic 1': 'software, software bug, bug, fixes, updates fixes, app, crashes, encountering software bug, encountering, encountering software',
 'Topic 2': 'data, deleted, files, lost, lost data, recover, accidentally deleted, important data, accidentally, deleted important data',
 'Topic 3': 'account, password, access account, access, account password, password reset, recover account, account password reset, password account, forgotten password',
 'Topic 4': 'networks, connecting, set, connect, fails connect, fails connect networks, set fails connect, set fails, connect networks, connect networks troubleshoot',
 'Topic 5': 'screen says, error message popping, popping screen, peculiar error, popping screen says, message popping screen, peculiar, message popping, peculiar error message, popping',
 'Topic 6': 'data safe, safe, concerned, co

In [34]:
bert_model.visualize_barchart()

* Intertopic Distance Map is generate to project the high dimensional document embeddings into a 2D space. 
* It organised the clusters into three distinct macro regions.
* The highly isolated space one in upper right quadrant isolates non technical administrative queries from primary twchnical corpus.
* The dense one in lower left encapsulates core engineering issues differentiating networking from broader softer and hardware issues

In [35]:
bert_model.visualize_topics()

* A cosine similarity matrix was generated across all topics.
* The predominance of low similarity scores validates the models high diversity score of 0.95, which confirms that the cluster represent distinct, non overlapping business issues.
* Strong cosine similarity is observed between topic 4 (Network setup) and topic 9 (Internet stability) which tends to preserve logical relationship
* Topic 0 exhibits moderate similarity across several clusters, which reflecting its role as foundational set

In [36]:
bert_model.visualize_heatmap()